In [ ]:
!pip install -q scrapy
#!pip install -q newspaper4k
!pip install -q newspaper3k
!pip install -q lxml_html_clean

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 817.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.2/311.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.8/259.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 50.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 6.8 MB/s eta 0:00:00


# Articulos de mapas de sitio usando scrapy

In [ ]:
!scrapy startproject news_scraper

New Scrapy project 'news_scraper', using template directory '/usr/local/lib/python3.11/dist-packages/scrapy/templates/project', created in:
    /content/news_scraper

You can start your first spider with:
    cd news_scraper
    scrapy genspider example example.com


## Extracción de URLs

In [ ]:
%cd news_scraper

/content/news_scraper


In [ ]:
%%writefile "news_extractor_spider.py"
import scrapy
from newspaper import build, Config, Article
import requests
from urllib.parse import urljoin, urlparse
import os
import time
import re
from datetime import datetime
import gzip
from scrapy.selector import Selector
import dateutil

class NewsUrlExtractorSpider(scrapy.Spider):
    name = 'news_extractor'

    def __init__(self, date="2025-01-01", *args, **kwargs):
        super(NewsUrlExtractorSpider, self).__init__(*args, **kwargs)
        self.from_date = datetime.strptime(date, '%Y-%m-%d').date()
        self.invalid_url_words = {'section', 'tag', 'template',
                                  'category', 'author', 'page-sitemap'
                                  'categories', 'video', 'image', 'temas',
								                  'live', 'microsite', 'focus', 'blog',
                                  'ocio', 'cine', 'board', 'character'}


    # Lista de páginas web
    start_urls=[
        'https://www.elcomercio.es/',
        'https://www.lne.es/',
        'https://www.lavanguardia.com/',
        'https://www.larazon.es/',
        'https://www.rtpa.es/',
        'https://www.europapress.es/',
        'https://www.20minutos.es/',
        'https://www.elperiodico.com/',
        'https://www.eldiario.es/',
        'https://www.elconfidencial.com/',
        'https://www.culturalgijonesa.org/',
        'https://www.elespanol.com/',
        'https://www.nortes.me/',
        'https://www.tribunasalamanca.com/',
        'https://migijon.com/',
        'https://www.telecinco.es/',
        'https://www.laprovincia.es/',
        'https://www.laopiniondemalaga.es/',
        'https://www.elfielato.es/',
        'https://www.teleprensa.com/',
        'https://www.infobae.com/',
        'https://www.asturiasmundial.com/', #newspaper
        'https://www.abc.es/', #newspaper
        'https://www.lavozdeasturias.es/', #newspaper
        'https://cualia.es/', #newspaper
        'https://www.lavozdegalicia.es/', # newspaper
        'http://www.gentedigital.es/', #newspaper
    ]

    # Metadata: nombre y sitemap
    metadata_urls={
        'https://www.elcomercio.es/': {'nombre': 'El Comercio',
                                       'sitemap': 'sitemap.xml'},
        'https://www.lne.es/': {'nombre': 'La Nueva España',
                                'sitemap': 'sitemap_google_news_52e19.xml'},
        'https://www.lavanguardia.com/': {'nombre': 'La Vanguardia',
                                          'sitemap': 'sitemap-google-news.xml'},
        'https://www.larazon.es/': {'nombre': 'La Razón',
                                    'sitemap': 'sitemaps/news.xml'},
        'https://www.rtpa.es/': {'nombre': 'Radiotelevisión del Principado de Asturias (RTPA)',
                                 'sitemap': 'sitemap-noticias.xml'},
        'https://www.europapress.es/': {'nombre': 'Europa Press'},
        'https://www.abc.es/': {'nombre': 'ABC'},
        'https://www.20minutos.es/': {'nombre': '20 Minutos',
                                      'sitemap': 'sitemap-google-news.xml'},
        'https://www.elperiodico.com/' : {'nombre': 'El Periódico',
                                          'sitemap': 'es/google-news.xml'},
        'https://www.eldiario.es/' : {'nombre': 'ElDiario.es',
                                      'sitemap': 'sitemap_google_news_25b87.xml'},
        'https://www.lavozdeasturias.es/': {'nombre': 'La Voz de Asturias'}, #newspaper
        'https://www.elconfidencial.com/': {'nombre': 'El Confidencial',
                                            'sitemap': 'newsitemap_4.xml'},
        'https://cualia.es/': {'nombre': 'Cualia'}, #newspaper
        'https://www.culturalgijonesa.org/': {'nombre': 'Cultural Gijonesa'},
        'https://www.elespanol.com/' : {'nombre': 'El Español',
                                        'sitemap': 'sitemap_google_news.xml'},
        'https://www.nortes.me/': {'nombre': 'Nortes'},
        'https://www.lavozdegalicia.es/': {'nombre': 'La Voz de Galicia'},
        'https://www.asturiasmundial.com/': {'nombre': 'Asturias Mundial'},
        'https://www.tribunasalamanca.com/': {'nombre': 'Tribuna Salamanca',
                                              'sitemap': 'sitemap_last_news.xml'},
        'https://migijon.com/': {'nombre': 'Mi Gijón',
                                 'sitemap': 'sitemap_index.xml'},
        'http://www.gentedigital.es/' : {'nombre': 'Gente Digital'}, #newspaper
        'https://www.infobae.com/' : {'nombre': 'Infobae',
                                      'sitemap': 'arc/outboundfeeds/news-sitemap2/'},
        'https://www.telecinco.es/' : {'nombre': 'Telecinco',
                                       'sitemap': 'sitemap_google_news.xml'},
        'https://www.laprovincia.es/' : {'nombre': 'La Provincia',
                                         'sitemap': 'sitemap_google_news_89aa6.xml'},
        'https://www.laopiniondemalaga.es/': {'nombre': 'La Opinión de Málaga',
                                              'sitemap': 'sitemap_google_news_fcad9.xml'},
        'https://www.elfielato.es/': {'nombre': 'El Fielato y El Nora',
                                      'sitemap': 'sitemap.news.xml'},
        'https://www.teleprensa.com/': {'nombre': 'Teleprensa',
                                        'sitemap': 'sitemap.news.xml'}
    }


    custom_settings = {
        'USER_AGENT': "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:98.0) Gecko/20100101 Firefox/98.0"
    }

    # Para extracción de xmls
    namespaces = {
        'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9',  # Default namespace
        'news': 'http://www.google.com/schemas/sitemap-news/0.9'
    }

    def get_base_url(self, url):
        parsed_url = urlparse(url)
        return f"{parsed_url.scheme}://{parsed_url.netloc}/"

    def normalize_date(self, date_string):
        try:
            # Conviertir la cadena de fechas en un objeto datetime
            parsed_date = dateutil.parser.parse(date_string)
            # Formatea el objeto datetime como 'YYYY-MM-DD'
            #return parsed_date.strftime('%Y-%m-%d')
            return parsed_date.date()
        except (ValueError, TypeError):
            raise ValueError(f"Formato de fecha inválido: '{date_string}'")

    def is_valid_xml_url(self, sitemap_url):
        for invalid_word in self.invalid_url_words:
            if invalid_word in sitemap_url:
                return False

        pattern = r'\b(1[0-9]{3}|20[0-1][0-9]|202[0-4])\b' # Matches 1000-1999, 2000-2019, and 2020-2024
        match_url = re.search(pattern, sitemap_url)
        if match_url:
            return False
        else:
            return True

    def parse(self, response):
        domain_base = self.get_base_url(response.url)
        # Empezar explorando los archivos xml predefinidos
        if domain_base in self.metadata_urls and 'sitemap' in self.metadata_urls[domain_base]:
            sitemap_name = self.metadata_urls[domain_base]['sitemap']
            #sitemap_name = self.sitemap_urls[domain]
            sitemap_url = urljoin(domain_base, sitemap_name)
            print(f"Accediendo al siguiente enlace especificado... {sitemap_url}")
            #yield scrapy.Request(sitemap_url, callback=self.parse_sitemap)
            yield scrapy.Request(sitemap_url, callback=lambda response: self.parse_sitemap(response, domain_base) )
        else:
            # Usar robots en caso no se tengas archivo xml predefinido
            robots_url = urljoin(response.url, "/robots.txt")
            print(f'Robots URL: {robots_url}')
            yield scrapy.Request(robots_url, callback=self.parse_robots, meta={'domain': response.url})

    def parse_robots(self, response):
        if response.status != 200:
            print(f"Error al acceder a la url: {response.status}")
            return
        domain = response.meta['domain']
        domain_base = self.get_base_url(domain)
        current_sitemap_urls = set()

        # Extraer urls de sitemaps de robots.txt
        for line in response.text.splitlines():
            if line.lower().startswith('sitemap:'):
                sitemap_url = line.split(':', 1)[1].strip()
                parsed_url = urlparse(sitemap_url)
                file_path = parsed_url.path
                file_name, file_extension = os.path.splitext(file_path)
                if file_extension.lower() == ".xml" and self.is_valid_xml_url(sitemap_url):
                    current_sitemap_urls.add(sitemap_url)

        if len(current_sitemap_urls) > 0:
            print(f"Se encontraron los siguientes enlaces xml: {current_sitemap_urls}")
            for sitemap_url in current_sitemap_urls:
                print(f"Accediendo al siguiente enlace... {sitemap_url}")
                #yield scrapy.Request(sitemap_url, callback=self.parse_sitemap)  # Llamada recursiva
                yield scrapy.Request(sitemap_url, callback=lambda response: self.parse_sitemap(response, domain) )
        else:
            print("No se encontraron enlaces xml en robots.txt. Accediendo a enlaces con librería Newspaper...")
            # Si no se encuentra ningún mapa del sitio,
            # utilizar Newspaper para obtener las URL de los artículos de noticias
            yield from self.get_news_urls(domain)

    def parse_sitemap(self, response, domain=None):

        # Sitemaps anidados. Tag <sitemap>
        for sitemap in response.xpath('//ns:sitemap', namespaces=self.namespaces):
            sitemap_loc = sitemap.xpath('./ns:loc/text()', namespaces=self.namespaces).get()

            # Omitir sitemaps antiguos
            last_mod = sitemap.xpath('./ns:lastmod/text()', namespaces=self.namespaces).get()
            if last_mod:
                lastmod = self.normalize_date(last_mod)
                if lastmod < self.from_date:
                    continue

            # Verificar que la url es un archivo xml
            parsed_url = urlparse(sitemap_loc)
            file_path = parsed_url.path
            file_name, file_extension = os.path.splitext(file_path)

            if file_extension.lower() == ".xml" and self.is_valid_xml_url(sitemap_loc):#'section' not in sitemap_loc:
                #print(f'Se encontró otro xml dentro del archivo actual: {sitemap_loc}') ## lots of outputs
                if sitemap_loc:
                    #yield scrapy.Request(sitemap_loc, callback=self.parse_sitemap)  # Llamada recursiva
                    yield scrapy.Request(sitemap_loc, callback=lambda response: self.parse_sitemap(response, domain) )
            elif file_extension.lower() == '.gz'  and self.is_valid_xml_url(sitemap_loc):#'section' not in sitemap_loc:
                print(f"Se encontró archivo comprimido {sitemap_loc}")
                #yield scrapy.Request(sitemap_loc, callback=self.parse_sitemap_gz)
                yield scrapy.Request(sitemap_loc, callback=lambda response: self.parse_sitemap_gz(response, domain) )


        # Extraer datos de cada tag <url>
        for url in response.xpath('//ns:url', namespaces=self.namespaces):

            loc = url.xpath('./ns:loc/text()', namespaces=self.namespaces).get()
            title = url.xpath('./news:news/news:title/text()', namespaces=self.namespaces).get()
            publication_date = url.xpath('./news:news/news:publication_date/text()', namespaces=self.namespaces).get()
            fuente = url.xpath('./news:news/news:publication/news:name/text()', namespaces=self.namespaces).get()

            if loc is None:
                break

            #if title is None or publication_date is None or loc is None:
            #    #self.logger.warning(f"Sitemap {response} contiene otra información. Omitiendo...")
            #    #print(f"Sitemap {response} no contiene la información requerida. Omitiendo...")
            #    #break
            title = "" if title is None else title
            if publication_date is None:
               publication_date = url.xpath('./ns:lastmod/text()', namespaces=self.namespaces).get()

            if publication_date is not None:
                publication_date_comp = self.normalize_date(publication_date)
                #publication_date_ = dateutil.parser.parse(publication_date).date()
                if publication_date_comp < self.from_date:
                    continue

            #if fuente is None:
            #    found_match = re.search(r"www\.(.*?)\.(.+)", loc)
            #    if found_match:
            #        fuente = found_match.group(1)

            if domain is not None:
                domain_base = self.get_base_url(domain)
                if domain_base in self.metadata_urls:
                    fuente = self.metadata_urls[domain_base]['nombre']
            #fuente = self.metadata_urls[domain]['nombre']
            yield {
                'fuente': fuente,
                'url': loc,
                'titulo': title,
                'fecha_publicacion': publication_date,
            }

    def parse_sitemap_gz(self, response, domain=None):

        compressed_file = response.body
        decompressed_file = gzip.decompress(compressed_file).decode("utf-8")
        selector = Selector(text=decompressed_file, type="xml")
        #yield from self.parse_sitemap(selector)
        yield from self.parse_sitemap(selector, domain)

    def get_news_urls(self, domain):

        config = Config()
        config.request_timeout = 5
        config.language= 'es'
        config.thread_timeout_seconds = 5
        config.memoize_articles = False
        config.fetch_images = False
        config.follow_meta_refresh = True
        config.number_threads = 4
        config.browser_user_agent = self.custom_settings['USER_AGENT']
        config.headers = {
            "User-Agent": self.custom_settings['USER_AGENT'],
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Accept-Encoding": "gzip, deflate",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
            "Sec-Fetch-Dest": "document",
            "Sec-Fetch-Mode": "navigate",
            "Sec-Fetch-Site": "none",
            "Sec-Fetch-User": "?1",
            "Cache-Control": "max-age=0",
        }
        try:
            print(f'Extrayendo noticias con librería Newspaper para url: {domain}')
            start_time = time.time()
            paper = build(domain, config=config)
            print(f"--- {(time.time() - start_time):.2f}s segundos ---")
            articulos_urls = set(paper.article_urls())
            print(f'Se encontraron {len(articulos_urls)} enlaces de noticias.')

            # Obtener fuente
            if domain is not None:
                domain_base = self.get_base_url(domain)
                if domain_base in self.metadata_urls:
                    fuente = self.metadata_urls[domain_base]['nombre']
                else:
                    found_match = re.search(r"www\.(.*?)\.(.+)", domain)
                    fuente = ''
                    if found_match:
                        fuente = found_match.group(1)


            titulo, fecha_publicacion = '', None
            for articulo_url in articulos_urls:
                yield {
                    'fuente': fuente,
                    'url': articulo_url,
                    'titulo': titulo,
                    'fecha_publicacion': fecha_publicacion,
                }
        except Exception as e:
            print(f"No se pudieron obtener noticias de {domain}")

Overwriting news_extractor_spider.py


Copiar archivo spider a carpeta para ejecutar

In [ ]:
!cp news_extractor_spider.py news_scraper/spiders

Eliminar csv para generar uno nuevo

In [ ]:
# Eliminar csv
import os
output_file_csv = "news_output.csv"
if os.path.exists(output_file_csv):
    os.remove(output_file_csv)

Ejecutar spyder de scrapy para gener el archivo csv con los enlaces

In [ ]:
!scrapy crawl news_extractor -o news_output.csv -s LOG_ENABLED=False -s ROBOTSTXT_OBEY=False -a date=2025-02-01
#!scrapy crawl news_extractor -o news_output.csv

Accediendo al siguiente enlace especificado... https://www.20minutos.es/sitemap-google-news.xml
Accediendo al siguiente enlace especificado... https://www.eldiario.es/sitemap_google_news_25b87.xml
Robots URL: https://www.nortes.me/robots.txt
Accediendo al siguiente enlace especificado... https://www.lne.es/sitemap_google_news_52e19.xml
Accediendo al siguiente enlace especificado... https://www.elcomercio.es/sitemap.xml
Accediendo al siguiente enlace especificado... https://www.lavanguardia.com/sitemap-google-news.xml
Accediendo al siguiente enlace especificado... https://www.laopiniondemalaga.es/sitemap_google_news_fcad9.xml
Accediendo al siguiente enlace especificado... https://www.elperiodico.com/es/google-news.xml
Se encontraron los siguientes enlaces xml: {'https://www.nortes.me/sitemap_index.xml'}
Accediendo al siguiente enlace... https://www.nortes.me/sitemap_index.xml
Accediendo al siguiente enlace especificado... https://www.elconfidencial.com/newsitemap_4.xml
Accediendo al sig

## Filtrar articulos por fecha

In [ ]:
import pandas as pd
from datetime import datetime
import numpy as np

# Cargar la data extraída
data_df = pd.read_csv("news_output.csv")

# Fecha inicial
fecha_inicial = "2025-2-18"

data_df['fecha_publicacion'] = pd.to_datetime(data_df['fecha_publicacion'], format='mixed', errors='coerce', utc=True)
data_df['fecha_publicacion'] = pd.to_datetime(data_df['fecha_publicacion'].dt.strftime("%Y-%m-%d"))
data_df = data_df[(data_df['fecha_publicacion'].between(fecha_inicial, datetime.now().strftime('%Y-%m-%d'))) | (pd.isna(data_df['fecha_publicacion']))]
data_df = data_df.sort_values(by='fecha_publicacion', ascending=False).drop_duplicates().reset_index(drop=True)
data_df

,fuente,url,titulo,fecha_publicacion
0,ElDiario.es,https://www.eldiario.es/castilla-la-mancha/soc...,Exigen ante el juzgado la clausura de los labo...,2025-02-21
1,Teleprensa,https://www.teleprensa.com/articulo/nacional-3...,El conductor que mató a una menor en Gurb (Bar...,2025-02-21
2,La Nueva España,https://www.lne.es/sociedad/2025/02/21/agenda-...,Agenda: qué hacer hoy 21 de febrero en Asturias,2025-02-21
3,La Nueva España,https://www.lne.es/economia/2025/02/21/comprav...,La compraventa de pisos se dispara el 18% en A...,2025-02-21
4,La Nueva España,https://www.lne.es/asturias-exterior/2025/02/2...,"Homenaje póstumo a Juan Velarde en Madrid: ""Er...",2025-02-21
...,...,...,...,...
12691,Gente Digital,http://www.gentedigital.es/getafe/noticia/3967...,NaN,NaT
12692,Gente Digital,http://www.gentedigital.es/pontevedra/noticia/...,NaN,NaT
12693,Gente Digital,http://www.gentedigital.es/palencia/noticia/40...,NaN,NaT
12694,Gente Digital,http://www.gentedigital.es/logrono/noticia/400...,NaN,NaT


In [ ]:
conteos = data_df['fuente'].value_counts()
conteos

,count
fuente,
La Voz de Galicia,1637
Gente Digital,1575
ABC,1226
La Razón,969
El Confidencial,958
La Vanguardia,916
El Español,840
20 Minutos,701
El Periódico,495


In [ ]:
len(data_df['fuente'].value_counts())

27

### Eliminar enlaces duplicados

In [ ]:
def remover_url_duplicados(df):
    df['non_null_count'] = df.notnull().sum(axis=1)
    df = df.sort_values(by=['url', 'non_null_count'], ascending=[True, False])
    df = df.drop_duplicates(subset=['url'], keep='first')
    df = df.drop(columns=['non_null_count'])
    df = df.sort_values(by='fecha_publicacion', ascending=False).reset_index(drop=True)
    return df

data_df = remover_url_duplicados(data_df)
data_df

,fuente,url,titulo,fecha_publicacion
0,Mi Gijón,https://migijon.com/about-us/,NaN,2025-02-21
1,La Opinión de Málaga,https://www.laopiniondemalaga.es/sucesos/2025/...,La Policía investiga la muerte de una niña en ...,2025-02-21
2,La Opinión de Málaga,https://www.laopiniondemalaga.es/sucesos/2025/...,Homicidios investiga si el exalcalde de Gandía...,2025-02-21
3,La Opinión de Málaga,https://www.laopiniondemalaga.es/sucesos/2025/...,El hombre y la mujer fallecidos en Córdoba mur...,2025-02-21
4,La Opinión de Málaga,https://www.laopiniondemalaga.es/sucesos/2025/...,ROBO ATRIO PELÍCULA | Hollywood ofrece a los l...,2025-02-21
...,...,...,...,...
12686,ABC,https://www.semana.es/familia-real-espanola/de...,NaN,NaT
12687,ABC,https://www.semana.es/familia-real-espanola/pr...,NaN,NaT
12688,ABC,https://www.semana.es/living/entramos-casa-osc...,NaN,NaT
12689,ABC,https://www.semana.es/moda/nagore-robles-a-ani...,NaN,NaT


In [ ]:
data_df['fuente'].value_counts()

,count
fuente,
La Voz de Galicia,1632
Gente Digital,1575
ABC,1226
La Razón,969
El Confidencial,958
La Vanguardia,916
El Español,840
20 Minutos,701
El Periódico,495


## Acceder al contenido de las noticias

### Extracción con request

In [ ]:
import requests
from bs4 import BeautifulSoup


# Lista de user-agents para probar
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
    "Mozilla/5.0 AppleWebKit/537.36 (KHTML, like Gecko; compatible; Googlebot/2.1; +http://www.google.com/bot.html) Chrome/W.X.Y.Z Safari/537.36"
]

def extraer_texto_articulos_request(url):
    try:
        # Primer intento sin user-agent
        response = requests.get(url)

        # Si la respuesta no es exitosa o el contenido está vacío, intente con agentes de usuario
        if response.status_code != 200 or not response.content:
            for user_agent in user_agents:
                headers = {
                    "User-Agent": user_agent,
                    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
                    "Accept-Language": "en-US,en;q=0.5",
                    "Accept-Encoding": "gzip, deflate",
                    "Connection": "keep-alive",
                }
                response = requests.get(url, headers=headers)

                if response.status_code == 200 and response.content:
                    #print("Encontrado con user-agent")
                    break

        html_str = response.content
        soup = BeautifulSoup(html_str, 'lxml')

        # Remover footer
        footer = soup.find('footer')
        if footer:
            footer.decompose()

        text_content = soup.find_all('p')

        text = ''
        for i in text_content:
            text += i.text.strip() + " "

        return text.strip() if text.strip() else None

    except Exception as e:
        print(f"Error in url: {url}, error: {e}")
        return None

In [ ]:
extraer_texto_articulos_request("https://cualia.es/carl-barks-el-creador-del-tio-gilito/")

'El encanto es la clave de la obra de Carl Barks (1901-2000). Con una calidad gráfica excepcional, este dibujante logró un completo equilibrio entre las convenciones del tebeo infantil y esa idea de la imaginación realizada\xa0que defiende Fernando Savater, y que, a lo largo de más de medio siglo, vino a ser la característica de publicaciones como Dumbo o Don Miki. En este aspecto, el atractivo de los cómics de Barks no necesita demasiadas explicaciones. Hay que embarcarse en su lectura, sin darle más vueltas, y reconocer en ellos esa diversión sin límites que uno experimenta en la infancia. Más allá de ese valor sentimental, la figura de Barks merece atención por razones que tienen que ver con el propio desarrollo de los estudios Disney. Ahora les explicaré por qué. En agosto de 1923 Walt y Roy Disney piden dinero a su tío Robert y con ese capital fundan los Estudios Disney. El equipo de animadores lo encabeza Ub Iwerks, a cuyas ordenes trabajan Fritz Freleng, Rudolph Ising, Les Clark

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

num_threads = 16

start_time = time.time()
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    data_df['texto'] = list(executor.map(extraer_texto_articulos_request, data_df['url']))
print(f"--- {(time.time() - start_time):.2f}s seconds ---")

Error in url: http://www.gentedigital.es/caceres/noticia/4008039/abre-sus-puertas-en-majadas-de-tietar-caceres-el-museo-de-historia-de-la-computacion-tras-casi-cuatro-anos-de-obras/, error: HTTPConnectionPool(host='www.gentedigital.es', port=80): Max retries exceeded with url: /caceres/noticia/4008039/abre-sus-puertas-en-majadas-de-tietar-caceres-el-museo-de-historia-de-la-computacion-tras-casi-cuatro-anos-de-obras/ (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x789359710950>, 'Connection to www.gentedigital.es timed out. (connect timeout=None)'))


<ipython-input-108-2b36f8adf2b6>:34: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_str, 'lxml')


--- 1448.13s seconds ---


In [ ]:
data_df

,fuente,url,titulo,fecha_publicacion,texto
0,Mi Gijón,https://migijon.com/about-us/,NaN,2025-02-21,"Pablo Cardona, semifinalista en el P1 de Riad ..."
1,La Opinión de Málaga,https://www.laopiniondemalaga.es/sucesos/2025/...,La Policía investiga la muerte de una niña en ...,2025-02-21,INVESTIGACIÓN Archivo - Coche de Policía Nacio...
2,La Opinión de Málaga,https://www.laopiniondemalaga.es/sucesos/2025/...,Homicidios investiga si el exalcalde de Gandía...,2025-02-21,INVESTIGACIÓN Perales Iborra Teresa Domínguez ...
3,La Opinión de Málaga,https://www.laopiniondemalaga.es/sucesos/2025/...,El hombre y la mujer fallecidos en Córdoba mur...,2025-02-21,Investigación José Antonio Aguilar Juan Pablo ...
4,La Opinión de Málaga,https://www.laopiniondemalaga.es/sucesos/2025/...,ROBO ATRIO PELÍCULA | Hollywood ofrece a los l...,2025-02-21,"1,6 MILLONES DE EUROS EN VINO EFE Luis Renduel..."
...,...,...,...,...,...
12686,ABC,https://www.semana.es/familia-real-espanola/de...,NaN,NaT,Isa Pantoja Admirador Reina Letizia Ausencia G...
12687,ABC,https://www.semana.es/familia-real-espanola/pr...,NaN,NaT,Isa Pantoja Admirador Reina Letizia Ausencia G...
12688,ABC,https://www.semana.es/living/entramos-casa-osc...,NaN,NaT,Isa Pantoja Admirador Reina Letizia Ausencia G...
12689,ABC,https://www.semana.es/moda/nagore-robles-a-ani...,NaN,NaT,Isa Pantoja Admirador Reina Letizia Ausencia G...


In [ ]:
data_df['texto'].iloc[-1]

'Newsletter Buscar en Turium NO TE PIERDAS\nEste pueblo blanco de Almería está entre los más bonitos de España Níjar no solo está en la lista de los Pueblos más bonitos de España, sino que además se ha convertido en uno de los destinos más visitados de Andalucía durante 2024. Lucía Lorenzo | 20 Feb 2025 España está llena de pueblos preciosos, donde las calles estrechas y empedradas llevan a quienes los recorren hasta pequeñas iglesias y joyas desconocidas. La belleza se multiplica cuando ponemos rumbo al sur y llegamos hasta Andalucía, donde los pueblos llenos de casas señoriales nos reciben con los brazos abiertos. Justo aquí, en la provincia de Almería, se encuentra uno de los destinos más bonitos: Níjar. Este municipio almeriense no solo se ha hecho con un merecido puesto en el listado de los Pueblos más bonitos de España, sino que se ha convertido en uno de los más turísticos de la región, con más de dos millones de visitantes durante el 2024. Algunos dicen que aquí se ocultan las 

In [ ]:
data_df[data_df['fuente'] == 'El Comercio'].iloc[0].values

array(['El Comercio',
       'https://www.elcomercio.es/real-aviles/ovetense-marcarle-real-oviedo-era-queria-tachar-20250221204111-nt.html',
       '«Soy ovetense y marcarle al Real Oviedo era algo que quería tachar de la lista»',
       Timestamp('2025-02-21 00:00:00'),
       "MiComercio Mis noticias Mis intereses Newsletters  Mi suscripción Mi cuenta Mis dispositivos Ayuda / Contacto El Club Descuentos Nuevo Cerrar sesión Buscar en El Comercio: Diario de Asturias Secciones Servicios Destacamos Es noticia Ampliar Alberto Santos Avilés Viernes, 21 de febrero 2025, 01:00 El Real Avilés Industrial Femenino aún no se ha repuesto de su 'semana grande' en la que tumbó a Sporting y Real Oviedo y ya  ... tiene que visitar mañana sábado (16 horas, La Planchada) a otro 'hueso' de la categoría como el Racing de Santander. Las jugadoras que entrena Pedro Arboleya afrontan esta cita con la moral por las nubes tras una excelente racha en la que destaca el olfato goleador de una de sus delanteras, 

In [ ]:
data_df[data_df['texto'].isna()]

,fuente,url,titulo,fecha_publicacion,texto
1051,El Español,https://www.elespanol.com/opinion/vinetas/2025...,El pensador de Podemos,2025-02-21,None
1581,Tribuna Salamanca,https://www.tribunasalamanca.com/noticias/3948...,Señal de televisión y telefonía para todos los...,2025-02-21,None
5184,El Fielato y El Nora,https://www.elfielato.es/articulo/el-nora/alum...,Los alumnos del CP San Cucao recaudan más de 1...,2025-02-20,None
7941,Gente Digital,http://www.gentedigital.es/caceres/noticia/400...,NaN,NaT,None
9334,La Voz de Galicia,https://edicionimpresa.lavozdegalicia.es/la-vo...,NaN,NaT,None
...,...,...,...,...,...
12643,La Voz de Galicia,https://www.lavozdegalicia.es/video/gradario/2...,NaN,NaT,None
12644,La Voz de Galicia,https://www.lavozdegalicia.es/video/gradario/2...,NaN,NaT,None
12645,La Voz de Galicia,https://www.lavozdegalicia.es/video/gradario/2...,NaN,NaT,None
12646,La Voz de Galicia,https://www.lavozdegalicia.es/video/torremarat...,NaN,NaT,None


In [ ]:
mascara_faltantes = data_df['texto'].isnull() | (data_df['texto'] == '')
datos_faltantes = data_df[mascara_faltantes]
print(f'Articulos que no pudo extraer texto: {len(datos_faltantes)}')

Articulos que no pudo extraer texto: 140


In [ ]:
datos_faltantes.values[:50]

array([['El Español',
        'https://www.elespanol.com/opinion/vinetas/20250221/pensador-podemos/925847407_19.html',
        'El pensador de Podemos', Timestamp('2025-02-21 00:00:00'), None],
       ['Tribuna Salamanca',
        'https://www.tribunasalamanca.com/noticias/394881/senal-de-television-y-telefonia-para-todos-los-ciudadanos-de-la-region',
        'Señal de televisión y telefonía para todos los ciudadanos de la región ',
        Timestamp('2025-02-21 00:00:00'), None],
       ['El Fielato y El Nora',
        'https://www.elfielato.es/articulo/el-nora/alumnos-cp-san-cucao-recaudan-mas-1000-euros-ela/20250220082316075483.html',
        'Los alumnos del CP San Cucao recaudan más de 1.000 euros contra la ELA',
        Timestamp('2025-02-20 00:00:00'), None],
       ['Gente Digital',
        'http://www.gentedigital.es/caceres/noticia/4008039/abre-sus-puertas-en-majadas-de-tietar-caceres-el-museo-de-historia-de-la-computacion-tras-casi-cuatro-anos-de-obras/',
        nan, NaT, N

## Filtrar textos

### Buscar textos que tengan al menos una palabra clave

In [ ]:
# Filtrar por palabras clave
keywords = ['universidad', 'oviedo']

# Regex pattern
pattern = '|'.join(keywords)

# Filtrar dataframe
filtered_df = data_df[data_df['texto'].str.contains(pattern, case=False, na=False)].reset_index(drop=True)

filtered_df

,fuente,url,titulo,fecha_publicacion,texto
0,La Opinión de Málaga,https://www.laopiniondemalaga.es/tendencias21/...,Descubren en Egipto la primera tumba real desd...,2025-02-21,Arqueología / Egiptología La tumba fue descubi...
1,La Opinión de Málaga,https://www.laopiniondemalaga.es/tendencias21/...,Un nuevo estudio comprueba una alarmante reduc...,2025-02-21,Ciencias de la Tierra Glaciares en las Montaña...
2,La Opinión de Málaga,https://www.laopiniondemalaga.es/nacional/2025...,DENUNCIAS DE ACOSO SEXUAL | El Gobierno evita ...,2025-02-21,DENUNCIAS DE ACOSO SEXUAL El fundador de Podem...
3,La Opinión de Málaga,https://www.laopiniondemalaga.es/opinion/2025/...,"López, Rodríguez, Navarro y asociados",2025-02-21,Opinión | La Libreta del Duque de Chantada Mel...
4,La Provincia,https://www.laprovincia.es/buzzeando/2025/02/2...,VIRAL INFLUENCER CANARIAS | Una influencer sob...,2025-02-21,C. P. L. Mar Molina El desconocimiento de aspe...
...,...,...,...,...,...
2585,La Voz de Galicia,https://www.lavozdegalicia.es/noticia/yes/2025...,NaN,NaT,Cuando Javier Cascón te cuenta su filosofía de...
2586,La Voz de Galicia,https://www.lavozdegalicia.es/noticia/yes/2025...,NaN,NaT,"Año 2024, primero de bachillerato en una clase..."
2587,La Voz de Galicia,https://www.lavozdegalicia.es/xlsemanal/cienci...,NaN,NaT,Récord submarino https://www.lavozdegalicia.es...
2588,La Voz de Galicia,https://www.lavozdegalicia.es/xlsemanal/natura...,NaN,NaT,Pon a prueba tu olfato https://www.lavozdegali...


In [ ]:
filtered_df['url'].values[0:100]

array(['https://www.laopiniondemalaga.es/tendencias21/2025/02/21/descubren-egipto-primera-tumba-real-114551152.html',
       'https://www.laopiniondemalaga.es/tendencias21/2025/02/21/nuevo-estudio-comprueba-alarmante-reduccion-114536622.html',
       'https://www.laopiniondemalaga.es/nacional/2025/02/21/gobierno-ahondar-caso-monedero-errejon-114541399.html',
       'https://www.laopiniondemalaga.es/opinion/2025/02/21/lopez-rodriguez-navarro-asociados-114524177.html',
       'https://www.laprovincia.es/buzzeando/2025/02/21/influencer-archipielago-islas-canarias-son-como-jackson-five-solo-te-sabes-dos-dv-114545469.html',
       'https://www.laopiniondemalaga.es/medio-ambiente/2025/02/21/millones-personas-queman-plastico-cocinar-114543379.html',
       'https://www.infobae.com/espana/agencias/2025/02/21/podemos-justifica-su-actuacion-ante-las-acusaciones-de-acoso-contra-monedero/',
       'https://www.infobae.com/colombia/2025/02/21/la-culpa-de-que-les-suban-las-cuotas-del-icetex-a-los-es

## Buscar textos que tengan todas las palabras clave

In [ ]:
# Lista de palabras clave
keywords = ['universidad', 'oviedo']

# Crear una condición de filtro para cada palabra clave
filter_condition = data_df['texto'].str.contains(keywords[0], case=False, na=False)

for keyword in keywords[1:]:
    filter_condition &= data_df['texto'].str.contains(keyword, case=False, na=False)

# Aplicar filtro
filtered_df_all = data_df[filter_condition].reset_index(drop=True)

filtered_df_all

,fuente,url,titulo,fecha_publicacion,texto
0,La Razón,https://www.larazon.es/galicia/quien-era-alfre...,"<![CDATA[¿Quién era Alfredo Brañas, el ejemplo...",2025-02-21,Galicia España Local Opinión Internacional Eco...
1,La Razón,https://www.larazon.es/asturias/fernando-laser...,"Fernando Laserna Cocina, nombrado nuevo Fiscal...",2025-02-21,Asturias España Local Opinión Internacional Ec...
2,Nortes,https://www.nortes.me/articulos-recientes/,NaN,2025-02-21,El general Augusto Pinochet quería ser un augu...
3,Nortes,https://www.nortes.me/2025/02/21/video-entrevi...,NaN,2025-02-21,El filósofo asturiano desglosa en esta convers...
4,Nortes,https://www.nortes.me/2025/02/21/una-estatua-p...,NaN,2025-02-21,Las políticas asturianas de memoria han sido a...
...,...,...,...,...,...
120,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/oviedo/...,NaN,NaT,Un total de 72 investigadoras de la Universida...
121,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/oviedo/...,NaN,NaT,La Consejería de Educación inaugurará el próxi...
122,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/oviedo/...,NaN,NaT,Andrew Silin y Kate Yakusheva se conocían de c...
123,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/siero/2...,NaN,NaT,"El alcalde de Siero, Ángel García, se ha mostr..."


In [ ]:
def mostrar_informacion(data_df):

    for index, fila in data_df.iterrows():
        if pd.isna(fila['fecha_publicacion']):
            fecha = 'Desconocido'
        else:
            fecha = fila['fecha_publicacion'].strftime('%Y/%m/%d')
        fuente = fila['fuente']

        titulo = 'Desconocido'
        if pd.notna(fila['titulo']):
            titulo = fila['titulo']

        print(titulo)
        print(fila['url'])
        print(f'{fecha} - {fuente}')
        print()

mostrar_informacion(filtered_df_all)

<![CDATA[¿Quién era Alfredo Brañas, el ejemplo que el PP pone como modelo de unidad y respeto?]]>
https://www.larazon.es/galicia/quien-era-alfredo-branas-ejemplo-que-pone-como-modelo-unidad-respeto-p7m_2025022167b868cc417ec20001013b82.html
2025/02/21 - La Razón

Fernando Laserna Cocina, nombrado nuevo Fiscal Delegado en la comunidad autónoma de Personas con Discapacidad y Mayores
https://www.larazon.es/asturias/fernando-laserna-cocina-nombrado-nuevo-fiscal-delegado-comunidad-autonoma-personas-discapacidad-mayores_2025022167b7b4e5500f9600011436eb.html
2025/02/21 - La Razón

Desconocido
https://www.nortes.me/articulos-recientes/
2025/02/21 - Nortes

Desconocido
https://www.nortes.me/2025/02/21/video-entrevista-con-juan-ponte-sobre-su-libro-el-capitalismo-no-existe/
2025/02/21 - Nortes

Desconocido
https://www.nortes.me/2025/02/21/una-estatua-para-el-rector-alas/
2025/02/21 - Nortes

La sierense Susana Madera será la nueva viceconsejera de Medio Ambiente
https://www.rtpa.es/noticias-astur